In [1]:
!pip install mysql-connector-python

  Using cached mysql_connector_python-9.7.0-py2.py3-none-any.whl.metadata (11 kB)
Using cached mysql_connector_python-9.7.0-py2.py3-none-any.whl (480 kB)


In [1]:
import json
import urllib.parse
import urllib.request
import mysql.connector
import os
from dotenv import load_dotenv

load_dotenv()


True

In [2]:
client_id = os.getenv('naver_client_id')
client_secret = os.getenv('naver_secret')
print(client_id, client_secret)

CUHEIpAbibYaQclD2Izd ZD7VAgaohF


In [17]:
searchText = urllib.parse.quote('이기적')
url = 'https://openapi.naver.com/v1/search/book.json?query=' + searchText + '&display=1'
request = urllib.request.Request(url)
request.add_header("X-Naver-Client-Id", client_id)
request.add_header("X-Naver-Client-Secret", client_secret)

In [18]:
response = urllib.request.urlopen(request)
response_body = response.read()
response_body = json.loads(response_body)
response_body

{'lastBuildDate': 'Tue, 30 Jun 2026 15:27:55 +0900',
 'total': 789,
 'start': 1,
 'display': 1,
 'items': [{'title': '이기적 유전자 (40주년 기념판)',
   'link': 'https://search.shopping.naver.com/book/catalog/32473030836',
   'image': 'https://shopping-phinf.pstatic.net/main_3247303/32473030836.20260331110653.jpg',
   'author': '리처드 도킨스',
   'discount': '18000',
   'publisher': '을유문화사',
   'pubdate': '20230130',
   'isbn': '9788932473901',
   'description': '독특한 발상과 놀라운 주장으로 40여 년간 수많은 찬사와 논쟁의 중심에 선 과학 교양서의 바이블!\n\n1976년, 처음 출간되었을 당시 과학계와 일반 대중들에게 폭발적인 반향을 불러일으키며 세기의 문제작으로 떠오른 『이기적 유전자』는 40년이라는 세월의 검증을 거치며 그 중요성과 깊이를 더욱더 확고하게 인정받았고, 25개 이상의 언어로 번역되었으며 젊은이들이 꼭 읽어야 할 과학계의 고전으로 자리 잡았다. 새로운 디자인과 휴대하기 좋은 판형으로 갈아 입은 이번 40주년 기념판에 새롭게 수록된 에필로그에서 저자는 여전히 ‘이기적 유전자’라는 개념이 갖고 있는 지속적인 타당성을 이야기하며 이 책이 전하는 메시지를 되새긴다.\n\n저자는 이 책에서 인간을 포함한 모든 생명체는 DNA 또는 유전자에 의해 창조된 생존 기계이며, 자기의 유전자를 후세에 남기려는 이기적인 행동을 수행하는 존재라고 주장한다. 이러한 주장은 생물학계를 비롯해 과학계를 떠들썩하게 만들었고, 40년 동안 학계와 언론의 수많은 찬사와 논쟁의 대상이 되었다. 저자는 자신의 주장을 뒷받침하기 위해서 성의 진

In [19]:
total_count = response_body['total']
start_num = 1
loop_count = total_count // 100 + 1
book_list = []

In [20]:
for i in range(loop_count):
    url = 'https://openapi.naver.com/v1/search/book.json?query=' + searchText + '&display=100' + f'&start={start_num}'
    request = urllib.request.Request(url)
    request.add_header("X-Naver-Client-Id", client_id)
    request.add_header("X-Naver-Client-Secret", client_secret)

    response = urllib.request.urlopen(request)
    response_body = response.read()
    response_body = json.loads(response_body)

    book_list += response_body['items']
    start_num += 100

    if start_num > 1000: break

print(len(book_list))

790


In [ ]:
from datetime import datetime

with mysql.connector.connect(
    host = 'localhost',
    user = 'capybara',
    password = '1234',
    database = 'bookdb'
) as connection:
    
    with connection.cursor() as cursor:
        sql = 'insert into naver_book' \
            '(book_title, book_image, author, publisher, isbn, book_description, pub_date)' \
            'values (%s, %s, %s, %s, %s, %s, %s)'

        for book_info in book_list:
            pub_str = book_info.get('pubdate', '')

            if pub_str:
                pub_date = datetime.strptime(pub_str, '%Y%m%d').date()
            else:
                pub_date = None
            values = (
                book_info['title'],
                book_info['image'],
                book_info['author'],
                book_info['publisher'],
                book_info['isbn'],
                book_info['description'],
                pub_date,
            )

            cursor.execute(sql, values)
            
    connection.commit()